In [0]:
-- Create External Volume for S3 Access (Unity Catalog approach)
-- Note: You need CREATE EXTERNAL VOLUME permission on the schema

CREATE EXTERNAL VOLUME IF NOT EXISTS `prism-sentinel-stream`.`prism_bronze`.`landing_volume`
LOCATION 's3://prism-sentinel-bronze-2026/landing/'
COMMENT 'External volume for landing zone data';

-- Verify the volume was created
DESCRIBE VOLUME `prism-sentinel-stream`.`prism_bronze`.`landing_volume`;

In [0]:
%python
       
# --- 01_BRONZE_GENERATOR.ipynb ---
from pyspark.sql.functions import col, expr, rand, when, current_timestamp, lit

# 1. PARAMETERS
NUM_ROWS = 20000000  # 20 Million Records
HIVE_TABLE = "`prism-sentinel-stream`.prism_bronze.transactions_raw"

# 2. GENERATE SKELETON
print(f"🏗️ Generating {NUM_ROWS:,} historical records...")
df_gen = spark.range(0, NUM_ROWS)

# 3. POPULATE WITH REALISTIC FINANCIAL DATA
# We include known evasion strings for the 'Evasion Hunter' to find later
countries = ["USA", "UK", "INDIA", "GERMANY", "IRAN_PROXY", "UAE", "SINGAPORE"]
counterparties = ["NORTH_STAR_SHIPPING", "N0RTH_ST4R_SH1PP1NG", "GLOBAL_LOGISTICS", "ACME_CORP", "SHELL_CO"]

# Convert Python lists to SQL array syntax
countries_sql = "array(" + ", ".join([f"'{c}'" for c in countries]) + ")"
counterparties_sql = "array(" + ", ".join([f"'{c}'" for c in counterparties]) + ")"

df_bronze = df_gen.withColumn("transaction_id", expr("uuid()")) \
    .withColumn("user_id", (rand() * 1000000).cast("int")) \
    .withColumn("amount", (rand() * 15000).cast("decimal(18,2)")) \
    .withColumn("country", expr(f"element_at({countries_sql}, int(rand() * {len(countries)}) + 1)")) \
    .withColumn("counterparty", expr(f"element_at({counterparties_sql}, int(rand() * {len(counterparties)}) + 1)")) \
    .withColumn("timestamp", current_timestamp()) \
    .withColumn("ingestion_source", lit("HISTORICAL_GENERATOR"))

# 4. PERSIST TO UNITY CATALOG TABLE
print(f"📦 Writing to Unity Catalog table {HIVE_TABLE}...")
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(HIVE_TABLE)

print("🔥 SUCCESS: Bronze 20M Baseline Created.")